# Can we predict a warranty claim within a year?
Target: `claim_within_year`. Plan: inspect data quality, establish a baseline, compare a few models with cross-validation, then evaluate once on a held-out test set. `device_serial` is an identifier and is excluded from features (checked below for any hidden signal).

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
from sklearn.inspection import permutation_importance
df = pd.read_csv('data.csv')
print(df.shape); print(df.isna().sum().to_dict()); print('duplicate serials:', df.device_serial.duplicated().sum())
df.describe().T

## Data quality and target

In [ ]:
y='claim_within_year'
print(df[y].value_counts(normalize=True))
print('daily_hours > 24:', (df.daily_hours>24).sum(), '| age < 0.05y:', (df.device_age_years<0.05).sum())
fig,ax=plt.subplots(1,3,figsize=(14,3.5))
for a,c in zip(ax,['device_age_years','price_eur','daily_hours']): df[c].hist(bins=50,ax=a); a.set_title(c)
plt.tight_layout(); plt.show()

`daily_hours` has values above 24, which is physically impossible for average daily use. I keep them (tree models are robust, and they are few) but flag them; they may be data-entry errors worth checking upstream.

## Univariate relationship with the target

In [ ]:
print(df.groupby('channel')[y].agg(['mean','size']))
for c in ['device_age_years','price_eur','daily_hours']:
    print(c); print(df.groupby(pd.qcut(df[c],5,duplicates='drop'),observed=True)[y].agg(['mean','size']).round(3))
print('serial vs target AUC:', round(roc_auc_score(df[y],df.device_serial),3))

## Models
Stratified 80/20 split; model comparison by 5-fold CV on the training part only. Metrics: ROC AUC and average precision (PR AUC), since claims are rare (~12.6%) and accuracy would be misleading.

In [ ]:
feats=['channel','device_age_years','price_eur','daily_hours']
X=df[feats]; t=df[y]
Xtr,Xte,ytr,yte=train_test_split(X,t,test_size=0.2,stratify=t,random_state=0)
pre=ColumnTransformer([('c',OneHotEncoder(),['channel']),('n',StandardScaler(),['device_age_years','price_eur','daily_hours'])])
models={
 'baseline (prior)':DummyClassifier(strategy='prior'),
 'logistic':Pipeline([('p',pre),('m',LogisticRegression(max_iter=1000))]),
 'grad. boosting':Pipeline([('p',ColumnTransformer([('c',OneHotEncoder(),['channel'])],remainder='passthrough')),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0))]),
}
cv=StratifiedKFold(5,shuffle=True,random_state=0)
rows=[]
for n,m in models.items():
    p=cross_val_predict(m,Xtr,ytr,cv=cv,method='predict_proba')[:,1]
    rows.append((n,roc_auc_score(ytr,p),average_precision_score(ytr,p)))
pd.DataFrame(rows,columns=['model','CV ROC AUC','CV avg precision']).round(3)

## Held-out test result

In [ ]:
rng=np.random.default_rng(0)
def boot(yt,p,n=500):
    yt=np.asarray(yt);r=[]
    for _ in range(n):
        i=rng.integers(0,len(yt),len(yt))
        if yt[i].min()!=yt[i].max(): r.append(roc_auc_score(yt[i],p[i]))
    return np.percentile(r,[2.5,97.5])
res={}
for n,m in models.items():
    m.fit(Xtr,ytr); p=m.predict_proba(Xte)[:,1]; res[n]=p
    lo,hi=boot(yte,p)
    print(f'{n:18s} AUC {roc_auc_score(yte,p):.3f} (95% CI {lo:.3f}-{hi:.3f})  AP {average_precision_score(yte,p):.3f}  [prevalence {yte.mean():.3f}]')
fig,ax=plt.subplots(1,2,figsize=(11,4))
for n,p in res.items():
    f,t_,_=roc_curve(yte,p); ax[0].plot(f,t_,label=n)
    pr,rc,_=precision_recall_curve(yte,p); ax[1].plot(rc,pr,label=n)
ax[0].plot([0,1],[0,1],'k:'); ax[0].set(xlabel='FPR',ylabel='TPR',title='ROC'); ax[1].set(xlabel='recall',ylabel='precision',title='Precision-recall'); ax[0].legend(); plt.tight_layout(); plt.show()

## What drives the prediction?

In [ ]:
m=models['grad. boosting']
pi=permutation_importance(m,Xte,yte,scoring='roc_auc',n_repeats=20,random_state=0)
print(pd.Series(pi.importances_mean,index=feats).sort_values(ascending=False).round(4))
p=res['grad. boosting']; q=pd.qcut(p,5,duplicates='drop')
pd.DataFrame({'y':yte.values,'p':p}).groupby(q,observed=True).agg(actual_rate=('y','mean'),mean_pred=('p','mean'),n=('y','size')).round(3)

## Conclusion
**Yes, moderately.** On the held-out test set the logistic regression reaches ROC AUC ~0.76 (95% CI ~0.72-0.80) and average precision ~0.34 versus 0.126 for the no-skill baseline; 5-fold CV on the training data gives a similar picture (AUC ~0.72). Gradient boosting does no better, so the signal is mostly simple and roughly monotone.

- Claim risk rises with **device age** (6% for under 2 years, 22% for the oldest quintile), **daily hours** (7% to 22%) and **price** (7% to 19%). Channel matters little (partner is slightly higher, ~14.6% vs ~11.5%).
- Calibration is good: in the top-risk quintile the actual rate is ~31% vs ~4% in the lowest, so the score is usable for ranking or pricing, but it is not precise enough to call individual claims. Many claimers still look low-risk.
- `device_serial` is not predictive (AUC 0.52, consistent with noise) and was excluded.

**Caveats:** 23 rows have `daily_hours` > 24 (impossible), which should be checked at the source. Data is a single snapshot, so no time-based validation was possible. The test set has ~1,400 rows (~176 claims), so metrics carry the uncertainty shown in the CIs. Age, hours and price are correlated with each other; the importance ranking should not be read as causal.